In [0]:
#Find the total number of matches and win count for each team.

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, lit, count

# Initialize Spark session
spark = SparkSession.builder.appName("TeamStats").getOrCreate()

# Sample data
data = [
    (1, "IND", "AUS", "IND"),
    (2, "IND", "NZ", "NZ"),
    (3, "AUS", "NZ", "AUS"),
    (4, "IND", "AUS", "IND"),
    (5, "NZ", "IND", "NZ")
]

# Create DataFrame
columns = ["id", "team1", "team2", "winner"]
df = spark.createDataFrame(data, columns)

# Calculate total matches for each team
team1_matches = df.select(col("team1").alias("team")).withColumn("match", lit(1))
team2_matches = df.select(col("team2").alias("team")).withColumn("match", lit(1))

total_matches = team1_matches.union(team2_matches) \
    .groupBy("team") \
    .agg(count("match").alias("total_matches"))

# Calculate wins for each team
wins = df.groupBy("winner").agg(count("winner").alias("wins")).withColumnRenamed("winner", "team")

# Join total_matches and wins
result = total_matches.join(wins, on="team", how="left").fillna(0)

# Show the result
result.show()


+----+-------------+----+
|team|total_matches|wins|
+----+-------------+----+
| IND|            4|   2|
| AUS|            3|   1|
|  NZ|            3|   2|
+----+-------------+----+

